# NF-v3 / CIC-IDS2017 TabPFN-v3 vs XGBoost -- Colab runner

Colab port of `tabpfn/nfv3_cic2018_multiclass_test.py`. Differences from the local script:

1. **Drive-backed I/O**: mounts Google Drive, reads data pkls from Drive, writes
   `per_class_metrics.csv` / `split_audit.csv` / `args.json` / `timings.json` back to Drive
   (mirrors the local script's `results/<ts>_<tag>/` layout), and additionally **saves the
   fitted XGBoost booster and TabPFN state to Drive** so a later session can reload them
   without re-fitting (TabPFN's `fit()` is cheap/lazy, but re-selecting/re-loading a 1-10M-row
   pkl each time is not -- see the reuse cell at the bottom).
2. **`--protocol reproduce` removed.** This project only uses the chronological-split +
   stratified-train-budget policy now (what used to be `--protocol sota`); the random-split /
   fixed-3,000-per-class path and the `--protocol` flag are gone. `--max-train-samples` is now
   the only train-budget knob.
3. **Per-class zero-P/R/F1 investigation baked in.** The 2026-08 sota runs
   (`results/20260806_201324_cic2017_full_tabpfn_multiclass/`,
   `results/20260810_091207_nfv3_cic2018_tabpfn_multiclass/`) had classes scoring exactly
   0.0/0.0/0.0 precision/recall/F1. Verified **not** an indexing bug (`class_names`/`y` are
   `LabelEncoder`-consistent for cic2017_full, and built from the same
   `{name: i for i, name in enumerate(class_names)}` mapping everywhere for cic2018 -- checked
   directly against `scripts/preprocess_cic2017_full_raw.py` and `exp_utils.labels_for`). Two
   distinct real causes instead:
   - **cic2017_full `bot` and `infiltration` (both TabPFN *and* XGBoost hit exactly 0):** real
     train/test feature-distribution shift from the per-class *chronological* split applied to
     a short, multi-phase attack capture. Directly measured: `bot`'s `Fwd/Bwd IAT Min` features
     differ by **>800x-7000x** in z-score between its train partition (chronologically-early
     rows within the single Friday-morning capture file) and test partition (chronologically-late
     rows in the same file) -- median |z| across all 70 features is 0.28 (bot) / 0.59
     (infiltration) vs 0.04-0.08 for well-behaved classes like `ddos`/`portscan`. Both models see
     a genuinely different population at test time than at train time; this is a split-methodology
     artifact of the dataset (Bot/Infiltration are multi-stage/non-stationary within their single
     capture window), not a code defect. The `diagnose_train_test_drift()` cell below reproduces
     this check for any run.
   - **cic2018 `web_attacks` (XGBoost only; TabPFN still got 0.56 F1):** plain starvation. That
     run used `--max-train-samples 100000` against a highly skewed natural pool (benign dominates
     at 10.5M), and the sota policy's *ratio-preserving* budget allocates web_attacks only
     `1522 * 100000/11069303 ~= 13` training rows. XGBoost's multiclass softmax can't carve out a
     region from 13 examples competing against classes with 10^5-10^7 examples; TabPFN's
     in-context few-shot design is comparatively more robust to this. Raise `max_train_samples`
     (this notebook now has GPU headroom to do that -- see below) or expect tiny classes to
     struggle at small budgets.

See `manuscript/report/0813.md` for the full writeup.

## Runtime setup

**Runtime > Change runtime type > GPU** -- pick the biggest GPU your Colab tier offers (A100 if
available). A T4 (16GB) or the free tier will only fit a few hundred thousand train rows; see the
GPU-memory-aware `max_train_samples` suggestion below (measured on an RTX 4090, 24GB: ~57,900
train rows per GB, confirmed linear up to ~1.37M rows with no plateau -- cic2018's full
12,069,313-row train pool would need ~209GB, which no single GPU has, Colab or otherwise. Use
`cic2018_capped` or a bounded `max_train_samples` for cic2018, not the raw uncapped pool).

In [ ]:
import subprocess
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"],
    capture_output=True, text=True,
).stdout)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# ---- Edit these two for your own Drive layout ----
DRIVE_ROOT = "/content/drive/MyDrive/imbal_cic_tabpfn"
TABPFN_COMMIT = "bc7b193cac5d81776585f8ca85952d2b903b467d"  # merge-base of the local tabpfn/ checkout with origin/main -- the local checkout's HEAD is one commit ahead but that commit only adds this test script/results/checkpoint, no src/ changes, so this is functionally identical package code AND (unlike local HEAD) actually exists on GitHub for pip to fetch

DATA_DIR = os.path.join(DRIVE_ROOT, "data")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
MODELS_DIR = os.path.join(DRIVE_ROOT, "saved_models")
CKPT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")     # the TabPFN foundation-model .ckpt (downloaded once)
RESUME_DIR = os.path.join(DRIVE_ROOT, "resume")         # mid-run fit checkpoints, keyed by config -- NOT the same thing as CKPT_DIR
for d in (DATA_DIR, RESULTS_DIR, MODELS_DIR, CKPT_DIR, RESUME_DIR):
    os.makedirs(d, exist_ok=True)

print("Upload whichever of these you plan to use to", DATA_DIR, "before running the experiment cell:")
print("  cic2017_full_raw.pkl                       (cic2017_full target, ~825MB)")
print("  nfv3_energy_suite_uncapped_scenarios.pkl    (cic2018 / bot_iot / ton_iot targets, FULL suite, ~14.6GB --")
print("                                               cic2018=20.1M, bot_iot=16.9M, ton_iot=27.5M rows; none of")
print("                                               these fit any single GPU uncapped)")
print("  nfv3_energy_suite_cic2018_scenarios.pkl     (cic2018_capped / bot_iot_capped / ton_iot_capped targets,")
print("                                               same file despite the name -- per-class-capped slices of")
print("                                               all 4 bundled datasets, ~443MB, easily fits)")

# --protocol reproduce removed 2026-08-13: this project only uses the
# chronological-split + stratified-train-budget policy now (formerly
# --protocol sota). max_train_samples is the only train-budget knob left.
CONFIG = dict(
    data=None,                      # None -> DATASET_CONFIG[target]["default_data"]
    test_cap_per_class=100_000,     # 0 = use all test rows
    test_batch_size=20_000,
    seed=42,
    device="auto",
    ignore_pretraining_limits=True, # required whenever max_train_samples is pushed past the checkpoint's own 1,000,000-row default -- see the GPU-aware suggestion cell
    max_train_samples=0,            # 0 here just means "not set yet"; filled in below from detected GPU memory, override freely
    n_estimators=4,
    xgb_n_estimators=300,
    xgb_max_depth=8,
    xgb_learning_rate=0.05,
    xgb_subsample=0.8,
    xgb_colsample_bytree=0.8,
    xgb_min_child_weight=1.0,
    xgb_reg_lambda=1.0,
)

In [ ]:
import subprocess, sys


def pip_install(*args):
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f"pip install {args} failed (see output above)")


try:
    import tabpfn
    print("tabpfn already importable (", tabpfn.__file__, ") -- skipping install")
except ImportError:
    pip_install(f"git+https://github.com/PriorLabs/tabpfn@{TABPFN_COMMIT}")
    print("Installed tabpfn @", TABPFN_COMMIT)

pip_install("xgboost", "huggingface_hub")
print("Installed xgboost + huggingface_hub")

In [ ]:
import json
import pickle
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import precision_recall_fscore_support

from tabpfn import TabPFNClassifier
from tabpfn.model_loading import save_fitted_tabpfn_model, load_fitted_tabpfn_model

In [ ]:
import shutil

CKPT_NAME = "tabpfn-v3-classifier-v3_20260417_multiclass.ckpt"
drive_ckpt_path = os.path.join(CKPT_DIR, CKPT_NAME)

if os.path.exists(drive_ckpt_path):
    print("Using cached checkpoint from Drive:", drive_ckpt_path)
else:
    try:
        from huggingface_hub import hf_hub_download
        print("Checkpoint not cached on Drive yet -- downloading from Hugging Face (Prior-Labs/tabpfn_3)...")
        local_path = hf_hub_download(repo_id="Prior-Labs/tabpfn_3", filename=CKPT_NAME)
        shutil.copy(local_path, drive_ckpt_path)
        print("Cached to Drive:", drive_ckpt_path)
    except Exception as e:
        print("Auto-download failed:", repr(e))
        print("Falling back to manual upload -- select the .ckpt file from your local tabpfn/ checkout:")
        from google.colab import files
        uploaded = files.upload()
        assert CKPT_NAME in uploaded, f"Expected an upload named {CKPT_NAME}, got {list(uploaded)}"
        shutil.move(CKPT_NAME, drive_ckpt_path)
        print("Saved uploaded checkpoint to Drive:", drive_ckpt_path)

CONFIG["model_path"] = drive_ckpt_path

## Helpers (copied from `scripts/exp_utils.py` so this notebook has no dependency on the rest of the repo)

In [ ]:
def load_pickle(path):
    with open(path, "rb") as handle:
        return pickle.load(handle)


def subset_indices(indices, maximum, seed):
    indices = np.asarray(indices, dtype=np.int64)
    if maximum <= 0 or len(indices) <= maximum:
        return indices
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(indices, maximum, replace=False))


def labels_for(indices, families, class_names):
    mapping = {name: i for i, name in enumerate(class_names)}
    return np.asarray([mapping[str(x)] for x in families[indices]], dtype=np.int64)


def scenario_chronological_split(indices, scenarios, timestamps):
    """Per-class(scenario) 60/20/20 split ordered by timestamp within each class."""
    split = {"train": [], "val": [], "test": []}
    audit = []
    for scenario in sorted(np.unique(scenarios[indices])):
        selected = indices[scenarios[indices] == scenario]
        ordered = selected[np.argsort(timestamps[selected], kind="stable")]
        n_train = int(len(ordered) * 0.6)
        n_val = int(len(ordered) * 0.2)
        n_test = len(ordered) - n_train - n_val
        if min(n_train, n_val, n_test) <= 0:
            raise ValueError(f"Scenario {scenario!r} is too small for 60/20/20")
        split["train"].extend(ordered[:n_train])
        split["val"].extend(ordered[n_train:n_train + n_val])
        split["test"].extend(ordered[n_train + n_val:])
        audit.append({
            "scenario": str(scenario), "total": len(ordered),
            "train": n_train, "val": n_val, "test": n_test,
            "train_last_timestamp": int(timestamps[ordered[n_train - 1]]),
            "val_first_timestamp": int(timestamps[ordered[n_train]]),
            "val_last_timestamp": int(timestamps[ordered[n_train + n_val - 1]]),
            "test_first_timestamp": int(timestamps[ordered[n_train + n_val]]),
        })
    return {
        name: np.sort(np.asarray(values, dtype=np.int64))
        for name, values in split.items()
    }, pd.DataFrame(audit)

## Dataset loaders (chronological split only -- the `reproduce`/random-split path is gone)

The NF-v3 "energy suite" pkls bundle *four* NetFlow-v3 datasets in one file, distinguished by
`dataset_names`: `cse_cic_ids2018` ("18"), `bot_iot`, `ton_iot`, and `unsw_nb15` (not wired up
here, add it the same way if you need it). `load_nfv3_suite_subset` is the shared loader --
`load_cic2018`/`load_bot_iot`/`load_ton_iot` are just it pinned to one `dataset_names` value, same
pattern as the original script's `load_cic2018`. All three are just as large as cic2018 in the
*uncapped* suite pkl (bot_iot ~16.9M rows, ton_iot ~27.5M rows -- even bigger than cic2018's
20.1M) so the same GPU-memory caveat applies; use the `_capped` variants (same pre-built
`nfv3_energy_suite_cic2018_scenarios.pkl`, which despite its name also holds capped bot_iot/
ton_iot/unsw_nb15 slices -- verified every class in it has enough rows for a 60/20/20 split,
smallest is bot_iot's `theft` at 1,615 rows) unless you've sized `max_train_samples` for your
GPU.

In [ ]:
def load_nfv3_suite_subset(args, dataset_name):
    suite = load_pickle(args.data)
    X = suite["X"]
    datasets = np.asarray(suite["dataset_names"])
    families = np.asarray(suite["families"])
    scenarios = np.asarray(suite["attack_scenarios"])
    timestamps = np.asarray(suite["timestamps"])

    target_idx = np.flatnonzero(datasets == dataset_name)
    if not len(target_idx):
        raise ValueError(f"No rows for dataset_names == {dataset_name!r}")

    class_names = sorted(np.unique(families[target_idx]).tolist())

    def label_fn(idx):
        return labels_for(idx, families, class_names)

    split, split_audit = scenario_chronological_split(target_idx, scenarios, timestamps)
    train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

    y_train_all = label_fn(train_idx)
    y_test_all = label_fn(test_idx)
    return X, class_names, train_idx, val_idx, test_idx, y_train_all, y_test_all, split_audit, label_fn


def load_cic2018(args):
    return load_nfv3_suite_subset(args, "cse_cic_ids2018")


def load_bot_iot(args):
    return load_nfv3_suite_subset(args, "bot_iot")


def load_ton_iot(args):
    return load_nfv3_suite_subset(args, "ton_iot")


def load_cic2017_full(args):
    d = load_pickle(args.data)
    X = d["X"]
    class_names = list(d["class_names"])
    y_full = np.asarray(d["y"], dtype=np.int64)
    all_idx = np.arange(len(y_full), dtype=np.int64)

    def label_fn(idx):
        return y_full[idx]

    time_proxy = np.asarray(d["time_proxy"], dtype=np.int64)
    split, split_audit = scenario_chronological_split(all_idx, y_full, time_proxy)
    split_audit = split_audit.rename(columns={"scenario": "class"})
    split_audit["class"] = split_audit["class"].astype(int).map(
        {i: name for i, name in enumerate(class_names)}
    )
    train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

    y_train_all = label_fn(train_idx)
    y_test_all = label_fn(test_idx)
    return X, class_names, train_idx, val_idx, test_idx, y_train_all, y_test_all, split_audit, label_fn


DATASET_CONFIG = {
    "cic2018": {
        "default_data": os.path.join(DATA_DIR, "nfv3_energy_suite_uncapped_scenarios.pkl"),
        "loader": load_cic2018,
        "tail_classes": ["bot", "infiltration", "web_attacks"],
        "out_tag": "nfv3_cic2018_tabpfn_multiclass",
    },
    "cic2018_capped": {
        "default_data": os.path.join(DATA_DIR, "nfv3_energy_suite_cic2018_scenarios.pkl"),
        "loader": load_cic2018,
        "tail_classes": ["bot", "infiltration", "web_attacks"],
        "out_tag": "nfv3_cic2018capped_tabpfn_multiclass",
    },
    "bot_iot": {
        "default_data": os.path.join(DATA_DIR, "nfv3_energy_suite_uncapped_scenarios.pkl"),
        "loader": load_bot_iot,
        "tail_classes": ["theft"],
        "out_tag": "nfv3_botiot_tabpfn_multiclass",
    },
    "bot_iot_capped": {
        "default_data": os.path.join(DATA_DIR, "nfv3_energy_suite_cic2018_scenarios.pkl"),
        "loader": load_bot_iot,
        "tail_classes": ["theft"],
        "out_tag": "nfv3_botiotcapped_tabpfn_multiclass",
    },
    "ton_iot": {
        "default_data": os.path.join(DATA_DIR, "nfv3_energy_suite_uncapped_scenarios.pkl"),
        "loader": load_ton_iot,
        "tail_classes": ["mitm", "ransomware"],
        "out_tag": "nfv3_toniot_tabpfn_multiclass",
    },
    "ton_iot_capped": {
        "default_data": os.path.join(DATA_DIR, "nfv3_energy_suite_cic2018_scenarios.pkl"),
        "loader": load_ton_iot,
        "tail_classes": ["mitm", "ransomware"],
        "out_tag": "nfv3_toniotcapped_tabpfn_multiclass",
    },
    "cic2017_full": {
        "default_data": os.path.join(DATA_DIR, "cic2017_full_raw.pkl"),
        "loader": load_cic2017_full,
        "tail_classes": [
            "heartbleed", "web-attack-sql-injection", "infiltration",
            "web-attack-xss", "web-attack-brute-force",
        ],
        "out_tag": "cic2017_full_tabpfn_multiclass",
    },
}

## GPU-memory-aware `max_train_samples` default

Empirical fit from a 5K-1M row sweep on a single RTX 4090 (24GB): peak CUDA memory during
`fit()`+`predict()` (n_estimators=4, ~1K test rows) grows **linearly**, `mem_GB ~= 0.23 +
rows/57,900` (R^2 ~ 1). Confirmed by direct probing that this is a real ceiling, not a small-N
artifact -- every point from 1.5M rows up OOM'd on the same 24GB card with no plateau. Full
cic2018 (12,069,313 train rows) would need ~209GB, unattainable on any single current GPU.
Treat the number below as a starting point and lower it if you still hit a CUDA OOM.

In [ ]:
ROWS_PER_GB = 57_900
BASELINE_GB = 1.0     # extra slack for CUDA context / test batching / n_estimators=4
SAFETY_MARGIN = 0.8

if torch.cuda.is_available():
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    suggested_max_train_samples = max(1, int((gpu_gb - BASELINE_GB) * ROWS_PER_GB * SAFETY_MARGIN))
else:
    gpu_gb = 0.0
    suggested_max_train_samples = 50_000
    print("WARNING: no GPU detected -- switch the Colab runtime to a GPU (Runtime > Change runtime type).")

print(f"Detected GPU memory: {gpu_gb:.1f} GB -> suggested max_train_samples ~= {suggested_max_train_samples:,}")
print("This is an extrapolation from a 24GB card, not a guarantee -- back off if you still hit a CUDA OOM.")

if CONFIG["max_train_samples"] == 0:
    CONFIG["max_train_samples"] = suggested_max_train_samples
    print("CONFIG[\'max_train_samples\'] set to the suggestion above; override manually for a fixed value.")

In [ ]:
def cap_per_class(indices, labels, n_classes, maximum, seed):
    if maximum <= 0:
        return np.sort(indices)
    chosen = []
    for i in range(n_classes):
        class_indices = indices[labels == i]
        if len(class_indices) == 0:
            continue
        chosen.append(subset_indices(class_indices, maximum, seed + 500 + i))
    return np.sort(np.concatenate(chosen))


def stratified_subset(indices, labels, n_classes, total_budget, seed):
    """Largest-remainder proportional subsample preserving each class's natural
    share; guarantees >=1 row for every present class."""
    indices = np.asarray(indices, dtype=np.int64)
    if total_budget <= 0 or total_budget >= len(indices):
        return np.sort(indices)
    counts = np.bincount(labels, minlength=n_classes)
    present = counts > 0
    n_present = int(present.sum())
    if total_budget < n_present:
        raise ValueError(
            f"max_train_samples={total_budget} is smaller than the number "
            f"of present classes ({n_present}); raise it to at least that."
        )
    raw = counts * total_budget / counts.sum()
    target = np.minimum(np.floor(raw).astype(np.int64), counts)
    target[present & (target < 1)] = 1
    remaining = total_budget - int(target.sum())
    if remaining > 0:
        headroom = counts - target
        for class_id in np.argsort(-(raw - np.floor(raw))):
            if remaining <= 0:
                break
            if headroom[class_id] <= 0:
                continue
            target[class_id] += 1
            headroom[class_id] -= 1
            remaining -= 1
    chosen = []
    for class_id in range(n_classes):
        if target[class_id] <= 0:
            continue
        class_indices = indices[labels == class_id]
        chosen.append(subset_indices(class_indices, int(target[class_id]), seed + class_id))
    return np.sort(np.concatenate(chosen))


def predict_in_batches(model, features, batch_size):
    if batch_size <= 0:
        raise ValueError("test_batch_size must be positive")
    n_rows = len(features)
    n_batches = (n_rows + batch_size - 1) // batch_size
    predictions = []
    for batch_number, start in enumerate(range(0, n_rows, batch_size), 1):
        stop = min(start + batch_size, n_rows)
        predictions.append(model.predict(features[start:stop]))
        if batch_number == 1 or batch_number % 10 == 0 or stop == n_rows:
            print(f"TabPFN predict batch {batch_number}/{n_batches}: rows {start:,}:{stop:,}")
    return np.concatenate(predictions)


def per_class_table(method, y_true, y_pred, class_names, tail_classes):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(class_names))), zero_division=0,
    )
    rows = [{
        "method": method, "class": name,
        "precision": precision[i], "recall": recall[i], "f1": f1[i],
        "support": int(support[i]),
    } for i, name in enumerate(class_names)]

    macro_f1 = float(np.mean(f1))
    weighted_f1 = float(np.average(f1, weights=support)) if support.sum() else 0.0
    tail_idx = [i for i, name in enumerate(class_names) if name in tail_classes]
    tail_f1 = float(np.mean(f1[tail_idx])) if tail_idx else float("nan")

    for label, value in [
        ("macro_avg", macro_f1), ("weighted_avg", weighted_f1), ("tail_avg", tail_f1),
    ]:
        rows.append({
            "method": method, "class": label,
            "precision": np.nan, "recall": np.nan, "f1": value,
            "support": int(support.sum()),
        })
    return rows

## Diagnostic: per-class train/test feature drift

This is what actually explains cic2017_full's `bot`/`infiltration` scoring exactly 0 P/R/F1 for
both models (see the top-of-notebook writeup) -- it is real distribution shift from the
chronological split, not an indexing bug. Low `train_n` alone (e.g. cic2018's `web_attacks` at a
small `max_train_samples`) is a separate, simpler starvation cause and shows up here as a small
`train_n` rather than a high `median_abs_z`.

In [ ]:
def diagnose_train_test_drift(X, class_names, train_idx, test_idx, label_fn):
    y_train = label_fn(train_idx)
    y_test = label_fn(test_idx)
    rows = []
    for cid, cname in enumerate(class_names):
        tr = train_idx[y_train == cid]
        te = test_idx[y_test == cid]
        if len(tr) == 0 or len(te) == 0:
            rows.append({"class": cname, "train_n": len(tr), "test_n": len(te), "median_abs_z": np.nan})
            continue
        Xtr = np.asarray(X[tr], dtype=np.float64)
        Xte = np.asarray(X[te], dtype=np.float64)
        std_tr = Xtr.std(axis=0) + 1e-6
        z = np.abs(Xtr.mean(axis=0) - Xte.mean(axis=0)) / std_tr
        rows.append({
            "class": cname, "train_n": len(tr), "test_n": len(te),
            "median_abs_z": float(np.median(z)),
        })
    return pd.DataFrame(rows).sort_values("median_abs_z", ascending=False, na_position="first")

## Mid-run checkpoint / resume

A full cic2017_full-scale TabPFN fit is estimated at several hours (see the runtime discussion
in chat) with no built-in progress reporting from inside `fit()` itself -- if the Colab session
dies mid-fit, that specific attempt is unrecoverable regardless of what we do (TabPFN's `fit()`
is one opaque call, no partial-progress hook). What we *can* do is checkpoint at the boundaries
`run_experiment` actually controls: right after TabPFN's `fit()` completes, right after its
`predict()` completes, and right after XGBoost's `fit()`+`predict()` complete. Each is saved to
`RESUME_DIR` under a filename derived only from what determines its *input* (dataset + train
selection + relevant hyperparameters, not test-eval-only settings like `test_cap_per_class`) --
re-running `run_experiment` with the same effective config finds the matching file and skips
straight past that step instead of redoing it. Pass `force_refit=True` to ignore any existing
checkpoint and redo everything from scratch.

In [ ]:
def resume_tag(target_dataset, args, train_used_n):
    return (f"{target_dataset}_mts{args.max_train_samples}_seed{args.seed}"
            f"_ne{args.n_estimators}_ipl{int(args.ignore_pretraining_limits)}_n{train_used_n}")

## Main experiment runner (Drive-backed results + saved models + resume)

In [ ]:
def run_experiment(target_dataset, overrides=None, force_refit=False):
    cfg = dict(CONFIG)
    cfg["target_dataset"] = target_dataset
    if overrides:
        cfg.update(overrides)
    if cfg.get("data") is None:
        cfg["data"] = DATASET_CONFIG[target_dataset]["default_data"]
    args = SimpleNamespace(**cfg)
    print(f"Args: {vars(args)}")

    tail_classes = DATASET_CONFIG[target_dataset]["tail_classes"]
    loader = DATASET_CONFIG[target_dataset]["loader"]
    X, class_names, train_idx, val_idx, test_idx, y_train_all, y_test_all, split_audit, label_fn = loader(args)
    print(f"target_dataset={target_dataset} classes ({len(class_names)}): {class_names}")

    train_counts = {name: int((y_train_all == i).sum()) for i, name in enumerate(class_names)}
    print(f"full train pool per class: {train_counts}")
    print(f"split rows: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")

    test_eval_idx = cap_per_class(test_idx, y_test_all, len(class_names), args.test_cap_per_class, args.seed + 900)
    y_test_eval = label_fn(test_eval_idx)
    print(f"test eval rows: {len(test_eval_idx)} (cap_per_class={args.test_cap_per_class})")
    X_test_eval = np.nan_to_num(np.asarray(X[test_eval_idx], dtype=np.float32))

    print("\n--- pre-fit diagnostic: train/test feature drift per class ---")
    drift_df = diagnose_train_test_drift(X, class_names, train_idx, test_idx, label_fn)
    print(drift_df.to_string(index=False))
    print("High median_abs_z (roughly >0.2-0.3 vs ~0.04-0.1 for well-behaved classes) means the")
    print("chronological split put a genuinely different-looking population of that class in train vs")
    print("test -- expect low recall regardless of model. Low train_n is a separate, simpler cause.\n")

    # Cap-selection is cheap/deterministic (numpy only, no GPU) -- always recompute it up front,
    # even on a resume, so resume_tag and train_used_counts/audit are always correct without
    # needing to serialize the index array itself.
    dummy_clf = TabPFNClassifier(
        device=args.device, model_path=args.model_path,
        ignore_pretraining_limits=args.ignore_pretraining_limits,
        inference_config={"SUBSAMPLE_SAMPLES": None}, random_state=args.seed,
        n_estimators=args.n_estimators, auto_scale_n_estimators=False,
    )
    max_samples = dummy_clf.get_inference_config().MAX_NUMBER_OF_SAMPLES
    print(f"TabPFN rows: full_train={len(train_idx):,} checkpoint_max={max_samples:,} n_estimators={args.n_estimators}")
    if not args.ignore_pretraining_limits and (
        args.max_train_samples <= 0 or args.max_train_samples > max_samples
    ) and len(train_idx) > max_samples:
        print(f"WARNING: max_train_samples={args.max_train_samples} will be silently re-capped to the "
              f"checkpoint's {max_samples:,}-row limit (uniformly, not stratified) because "
              "ignore_pretraining_limits was not set.")

    train_used_idx = train_idx
    cap_policy_applied = "none"
    if args.max_train_samples > 0 and args.max_train_samples < len(train_used_idx):
        pre_cap_labels = label_fn(train_used_idx)
        train_used_idx = stratified_subset(
            train_used_idx, pre_cap_labels, len(class_names), args.max_train_samples, args.seed + 850,
        )
        cap_policy_applied = "stratified_ratio_preserving"

    if not args.ignore_pretraining_limits and len(train_used_idx) > max_samples:
        train_used_idx = subset_indices(train_used_idx, max_samples, args.seed + 800)

    train_was_capped = len(train_used_idx) < len(train_idx)
    y_train_used = label_fn(train_used_idx)
    train_used_counts = {name: int((y_train_used == i).sum()) for i, name in enumerate(class_names)}
    print(f"TabPFN/XGBoost train selection: used={len(train_used_idx):,} capped={train_was_capped} "
          f"cap_policy_applied={cap_policy_applied} max_train_samples={args.max_train_samples} "
          f"per_class={train_used_counts}")

    train_used_audit = pd.DataFrame([
        {"split": "train_used", "class": name, "count": count}
        for name, count in train_used_counts.items()
    ])
    split_audit = pd.concat([split_audit, train_used_audit], ignore_index=True, sort=False)

    X_train_used = np.nan_to_num(np.asarray(X[train_used_idx], dtype=np.float32))

    tag = resume_tag(target_dataset, args, len(train_used_idx))
    tabpfn_ckpt = os.path.join(RESUME_DIR, f"{tag}_tabpfn.tabpfn_fit")
    tabpfn_pred_ckpt = os.path.join(RESUME_DIR, f"{tag}_tabpfn_pred.npy")
    xgb_ckpt = os.path.join(RESUME_DIR, f"{tag}_xgboost.json")

    all_rows, timings = [], []

    # ---- TabPFN: resume fit if checkpointed, else fit + checkpoint immediately ----
    if not force_refit and os.path.exists(tabpfn_ckpt):
        print(f"Resuming TabPFN from checkpoint: {tabpfn_ckpt} (skipping the multi-hour fit())")
        clf = load_fitted_tabpfn_model(tabpfn_ckpt, device=args.device)
        tabpfn_fit_seconds = None
    else:
        clf = dummy_clf
        t0 = time.time()
        clf.fit(X_train_used, y_train_used)
        tabpfn_fit_seconds = time.time() - t0
        save_fitted_tabpfn_model(clf, tabpfn_ckpt)
        print(f"TabPFN fit done in {tabpfn_fit_seconds:.1f}s -- checkpointed to {tabpfn_ckpt} "
              "(a resumed re-run will skip straight past this step)")

    if not force_refit and os.path.exists(tabpfn_pred_ckpt):
        print(f"Resuming TabPFN predictions from checkpoint: {tabpfn_pred_ckpt}")
        y_pred_tabpfn = np.load(tabpfn_pred_ckpt)
    else:
        t0 = time.time()
        y_pred_tabpfn = predict_in_batches(clf, X_test_eval, args.test_batch_size)
        tabpfn_predict_seconds = time.time() - t0
        np.save(tabpfn_pred_ckpt, y_pred_tabpfn)
        print(f"TabPFN predict done in {tabpfn_predict_seconds:.1f}s -- checkpointed to {tabpfn_pred_ckpt}")
    all_rows.extend(per_class_table("tabpfn_v3", y_test_eval, y_pred_tabpfn, class_names, tail_classes))

    # ---- XGBoost: same resume pattern (fit is fast, but free to checkpoint too) ----
    if not force_refit and os.path.exists(xgb_ckpt):
        print(f"Resuming XGBoost from checkpoint: {xgb_ckpt}")
        booster = xgb.XGBClassifier()
        booster.load_model(xgb_ckpt)
    else:
        booster = xgb.XGBClassifier(
            n_estimators=args.xgb_n_estimators, max_depth=args.xgb_max_depth,
            learning_rate=args.xgb_learning_rate, subsample=args.xgb_subsample,
            colsample_bytree=args.xgb_colsample_bytree, min_child_weight=args.xgb_min_child_weight,
            reg_lambda=args.xgb_reg_lambda, objective="multi:softprob", num_class=len(class_names),
            eval_metric="mlogloss", n_jobs=-1, random_state=args.seed,
        )
        booster.fit(X_train_used, y_train_used)
        booster.save_model(xgb_ckpt)
        print(f"XGBoost fit done -- checkpointed to {xgb_ckpt}")
    y_pred_xgb = booster.predict(X_test_eval)
    all_rows.extend(per_class_table("xgboost", y_test_eval, y_pred_xgb, class_names, tail_classes))

    timings.append({
        "target_dataset": target_dataset,
        "train_pool_rows": len(train_idx), "train_rows": len(train_used_idx),
        "train_was_capped": train_was_capped, "max_train_samples": args.max_train_samples,
        "train_cap_policy_applied": cap_policy_applied,
        "validation_rows_unused": len(val_idx), "test_pool_rows": len(test_idx),
        "test_evaluation_rows": len(test_eval_idx), "test_cap_per_class": args.test_cap_per_class,
        "tabpfn_test_batch_size": args.test_batch_size, "tabpfn_checkpoint_max_samples": max_samples,
        "tabpfn_n_estimators": args.n_estimators,
        "tabpfn_fit_seconds": tabpfn_fit_seconds, "resume_tag": tag,
    })

    table = pd.DataFrame(all_rows)
    summary = table[table["class"].isin(["macro_avg", "weighted_avg", "tail_avg"])]
    print("\n=== summary (macro / weighted / tail F1) ===")
    print(summary.pivot(index="class", columns="method", values="f1").to_string())

    # ---- save results to Drive ----
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_tag = DATASET_CONFIG[target_dataset]["out_tag"]
    out_dir = os.path.join(RESULTS_DIR, f"{ts}_{out_tag}")
    os.makedirs(out_dir, exist_ok=True)
    table.to_csv(os.path.join(out_dir, "per_class_metrics.csv"), index=False)
    split_audit.to_csv(os.path.join(out_dir, "split_audit.csv"), index=False)
    drift_df.to_csv(os.path.join(out_dir, "train_test_drift_diagnostic.csv"), index=False)
    with open(os.path.join(out_dir, "args.json"), "w", encoding="utf-8") as handle:
        json.dump(vars(args), handle, indent=2)
    with open(os.path.join(out_dir, "timings.json"), "w", encoding="utf-8") as handle:
        json.dump(timings, handle, indent=2)
    print(f"\nWrote results to {out_dir}")

    # ---- copy the (already-checkpointed) fitted models into the timestamped saved_models/ record ----
    model_tag = f"{ts}_{out_tag}"
    xgb_path = os.path.join(MODELS_DIR, f"{model_tag}_xgboost.json")
    shutil.copy(xgb_ckpt, xgb_path)
    print(f"Saved XGBoost model: {xgb_path}")

    tabpfn_path = os.path.join(MODELS_DIR, f"{model_tag}_tabpfn.tabpfn_fit")
    shutil.copy(tabpfn_ckpt, tabpfn_path)
    print(f"Saved fitted TabPFN state: {tabpfn_path}")
    print("Reload either of these later with:")
    print(f"  booster2 = xgb.XGBClassifier(); booster2.load_model({xgb_path!r})")
    print(f"  clf2 = load_fitted_tabpfn_model({tabpfn_path!r}, device={args.device!r})")

    return {"table": table, "split_audit": split_audit, "drift": drift_df, "timings": timings, "out_dir": out_dir}

## Run it

Available `target_dataset` values: `"cic2017_full"`, `"cic2018"` / `"cic2018_capped"`,
`"bot_iot"` / `"bot_iot_capped"`, `"ton_iot"` / `"ton_iot_capped"`.

`"cic2017_full"` fits comfortably on a Colab A100 (~30GB estimated). For the NF-v3 suite
datasets, prefer the `_capped` variant (all trivially fit -- largest is ton_iot_capped at 810K
rows) unless you've deliberately raised `max_train_samples` and confirmed your GPU can take it --
the raw/uncapped `cic2018`/`bot_iot`/`ton_iot` targets are all 17-27M rows and none of their full
train pools fit on any single GPU (see the writeup at the top).

In [ ]:
result = run_experiment("cic2017_full")
# result = run_experiment("cic2018_capped")
# result = run_experiment("bot_iot_capped")
# result = run_experiment("ton_iot_capped")
# result = run_experiment("cic2018")    # only with a deliberately-set/lowered max_train_samples
# result = run_experiment("bot_iot")    # only with a deliberately-set/lowered max_train_samples
# result = run_experiment("ton_iot")    # only with a deliberately-set/lowered max_train_samples

## Reusing a saved model in a later session

Re-run the setup cells above (Drive mount, config, pip installs, checkpoint cell, helper/loader
cells) to get `xgb`, `TabPFNClassifier`, `load_fitted_tabpfn_model` back, then:

```python
booster2 = xgb.XGBClassifier()
booster2.load_model("/content/drive/MyDrive/imbal_cic_tabpfn/saved_models/<tag>_xgboost.json")

clf2 = load_fitted_tabpfn_model(
    "/content/drive/MyDrive/imbal_cic_tabpfn/saved_models/<tag>_tabpfn.tabpfn_fit",
    device="cuda",
)
# clf2 is ready to call .predict()/.predict_proba() immediately -- no re-fit needed.
```

`save_fitted_tabpfn_model` stores the fitted state (init params + the executor/KV-cache state)
but *not* the foundation model weights, so it stays small; `load_fitted_tabpfn_model` re-attaches
the checkpoint weights on load.